### **Configure Silver notebook**

In [1]:
BRONZE_LAKEHOUSE_PATH = (
    "abfss://Banking_Fraud_Analytics@onelake.dfs.fabric.microsoft.com/"
    "banking_bronze_lh.Lakehouse"
)

BRONZE_DBO_PATH = (
    f"{BRONZE_LAKEHOUSE_PATH}/Tables/dbo"
)

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 3, Finished, Available, Finished, False)

In [2]:
from pyspark.sql import functions as F

AUDIT_TABLE_PATH = (
    f"{BRONZE_LAKEHOUSE_PATH}/Tables/dbo/bronze_ingestion_audit"
)

def get_latest_batch_id() -> str:
    """Return the latest batch where all nine datasets loaded successfully."""

    audit_df = (
        spark.read
        .format("delta")
        .load(AUDIT_TABLE_PATH)
    )

    latest_batch = (
        audit_df
        .filter(
            (F.col("load_status") == "SUCCESS") &
            (F.col("batch_id").isNotNull()) &
            (F.trim(F.col("batch_id")) != "")
        )
        .groupBy("batch_id")
        .agg(
            F.countDistinct("table_name").alias("successful_tables")
        )
        .filter(F.col("successful_tables") == 9)
        .orderBy(F.col("batch_id").desc())
        .first()
    )

    if latest_batch is None:
        raise ValueError("No complete successful Bronze batch found.")

    return latest_batch["batch_id"]


LATEST_BATCH_ID = get_latest_batch_id()

BRONZE_BASE_PATH = f"{BRONZE_LAKEHOUSE_PATH}/Files/Source"
BATCH_PATH = f"{BRONZE_BASE_PATH}/batch_id={LATEST_BATCH_ID}"

print("Latest successful batch:", LATEST_BATCH_ID)
print("Bronze batch path:", BATCH_PATH)

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 4, Finished, Available, Finished, False)

Latest successful batch: 20261107_001955
Bronze batch path: abfss://Banking_Fraud_Analytics@onelake.dfs.fabric.microsoft.com/banking_bronze_lh.Lakehouse/Files/Source/batch_id=20261107_001955


### **load raw banking datasets**

In [3]:
from pyspark.sql import DataFrame

# Dataset configuration
BERKA_TABLES = {
    "account": "account.csv",
    "card": "card.csv",
    "client": "client.csv",
    "disp": "disp.csv",
    "district": "district.csv",
    "loan": "loan.csv",
    "orders": "order.csv",
    "trans": "trans.csv"
}


# CSV reader
def read_bronze_csv(file_path: str, delimiter: str = ";") -> DataFrame:
    """Read a raw CSV file from the Bronze Lakehouse."""

    return (
        spark.read
        .format("csv")
        .option("header", "true")
        .option("sep", delimiter)
        .option("inferSchema", "false")
        .option("mode", "PERMISSIVE")
        .load(file_path)
    )


# Read Berka datasets
bronze_dfs = {}

BERKA_PATH = f"{BATCH_PATH}/Berka-Czech-bank"

for table_name, file_name in BERKA_TABLES.items():

    file_path = f"{BERKA_PATH}/{file_name}"

    bronze_dfs[table_name] = read_bronze_csv(
        file_path,
        delimiter=";"
    )

    print(f"Loaded Berka dataset: {table_name}")


# Read PaySim dataset
PAYSIM_PATH = f"{BATCH_PATH}/PaySim/paysim.csv"

bronze_dfs["paysim"] = read_bronze_csv(
    PAYSIM_PATH,
    delimiter=","
)

print("Loaded PaySim dataset")


# Verify datasets
print("\nDataset row counts:")

for table_name, df in bronze_dfs.items():
    print(f"{table_name}: {df.count():,}")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 5, Finished, Available, Finished, False)

Loaded Berka dataset: account
Loaded Berka dataset: card
Loaded Berka dataset: client
Loaded Berka dataset: disp
Loaded Berka dataset: district
Loaded Berka dataset: loan
Loaded Berka dataset: orders
Loaded Berka dataset: trans
Loaded PaySim dataset

Dataset row counts:
account: 4,500
card: 892
client: 5,369
disp: 5,369
district: 77
loan: 682
orders: 6,471
trans: 1,056,320
paysim: 6,362,620


### **Data quality profiling**

In [4]:
# Data quality configuration

PRIMARY_KEYS = {
    "account": ["account_id"],
    "card": ["card_id"],
    "client": ["client_id"],
    "disp": ["disp_id"],
    "district": ["district_id"],
    "loan": ["loan_id"],
    "orders": ["order_id"],
    "trans": ["trans_id"]
}


# Check missing values

def profile_missing_values(df: DataFrame) -> None:
    """Count NULL and blank values in each column."""

    missing_counts = [
        F.count(
            F.when(
                F.col(column).isNull() |
                (F.trim(F.col(column)) == ""),
                column
            )
        ).alias(column)
        for column in df.columns
    ]

    df.agg(*missing_counts).show(truncate=False)


# Check duplicate primary keys

def profile_duplicates(df: DataFrame,table_name: str,key_columns: list) -> None:
    """Identify duplicate primary key values."""

    duplicate_count = (
        df.groupBy(*key_columns)
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{table_name:<12} "
        f"Duplicate key groups: {duplicate_count}"
    )


# Run data quality profiling

for table_name, df in bronze_dfs.items():

    print(f"\n{'=' * 50}")
    print(f"DATA QUALITY: {table_name.upper()}")
    print(f"{'=' * 50}")

    profile_missing_values(df)

    if table_name in PRIMARY_KEYS:
        profile_duplicates(
            df,
            table_name,
            PRIMARY_KEYS[table_name]
        )
    else:
        print("No primary key defined for this dataset.")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 6, Finished, Available, Finished, False)


DATA QUALITY: ACCOUNT
+----------+-----------+---------+----+
|account_id|district_id|frequency|date|
+----------+-----------+---------+----+
|0         |0          |0        |0   |
+----------+-----------+---------+----+

account      Duplicate key groups: 0

DATA QUALITY: CARD
+-------+-------+----+------+
|card_id|disp_id|type|issued|
+-------+-------+----+------+
|0      |0      |0   |0     |
+-------+-------+----+------+

card         Duplicate key groups: 0

DATA QUALITY: CLIENT
+---------+------------+-----------+
|client_id|birth_number|district_id|
+---------+------------+-----------+
|0        |0           |0          |
+---------+------------+-----------+

client       Duplicate key groups: 0

DATA QUALITY: DISP
+-------+---------+----------+----+
|disp_id|client_id|account_id|type|
+-------+---------+----------+----+
|0      |0        |0         |0   |
+-------+---------+----------+----+

disp         Duplicate key groups: 0

DATA QUALITY: DISTRICT
+-----------+-----------

### **Inspect missing transaction fields**

In [5]:
# Inspect missing transaction fields

print("TRANSACTION TABLE")
print("Missing values by column")

trans_df = bronze_dfs["trans"]

trans_df.select(
    F.count(
        F.when(F.col("operation").isNull(), 1)
    ).alias("missing_operation"),

    F.count(
        F.when(F.col("k_symbol").isNull(), 1)
    ).alias("missing_k_symbol"),

    F.count(
        F.when(F.col("bank").isNull(), 1)
    ).alias("missing_bank"),

    F.count(
        F.when(F.col("account").isNull(), 1)
    ).alias("missing_account")
).show()


# Inspect transaction operation values

print("Transaction operation values")

display(
    trans_df
    .groupBy("operation")
    .count()
    .orderBy(F.desc("count"))
)


# Inspect transaction symbols

print("Transaction symbol values")

display(
    trans_df
    .groupBy("k_symbol")
    .count()
    .orderBy(F.desc("count"))
)


# Inspect transaction types

print("Transaction types")

display(
    trans_df
    .groupBy("type")
    .count()
    .orderBy(F.desc("count"))
)

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 7, Finished, Available, Finished, False)

TRANSACTION TABLE
Missing values by column
+-----------------+----------------+------------+---------------+
|missing_operation|missing_k_symbol|missing_bank|missing_account|
+-----------------+----------------+------------+---------------+
|           183114|          481881|      782812|         760931|
+-----------------+----------------+------------+---------------+

Transaction operation values


SynapseWidget(Synapse.DataFrame, 72565576-7c50-43dd-a12b-84181ca99902)

Transaction symbol values


SynapseWidget(Synapse.DataFrame, d48fa6b6-d2b1-4778-ab40-8206d0401016)

Transaction types


SynapseWidget(Synapse.DataFrame, 53af375a-21bd-4c6e-a548-99b81142439d)

### **Standardise Column Names**

In [6]:
# Standardise text columns

def standardise_text_values(df: DataFrame) -> DataFrame:
    """Trim text values and convert blank strings to NULL."""

    for column in df.columns:

        df = df.withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "",
                F.lit(None).cast("string")
            ).otherwise(
                F.trim(F.col(column))
            )
        )

    return df


# Apply standardisation to all datasets

cleaned_dfs = {}

for table_name, df in bronze_dfs.items():

    cleaned_df = standardise_text_values(df)

    cleaned_dfs[table_name] = cleaned_df

    print(f"Standardised: {table_name}")


# Verify transaction fields

display(
    cleaned_dfs["trans"]
    .agg(
        F.count(
            F.when(F.col("operation").isNull(), 1)
        ).alias("missing_operation"),

        F.count(
            F.when(F.col("k_symbol").isNull(), 1)
        ).alias("missing_k_symbol"),

        F.count(
            F.when(F.col("bank").isNull(), 1)
        ).alias("missing_bank"),

        F.count(
            F.when(F.col("account").isNull(), 1)
        ).alias("missing_account")
    )
)

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 8, Finished, Available, Finished, False)

Standardised: account
Standardised: card
Standardised: client
Standardised: disp
Standardised: district
Standardised: loan
Standardised: orders
Standardised: trans
Standardised: paysim


SynapseWidget(Synapse.DataFrame, d729553c-d1ee-4ab2-9fa8-37330a817064)

### **Data Cleaning & Data Convertion**

In [7]:
# Date conversion function

def convert_berka_date(column_name: str):
    """Convert YYMMDD values into Spark dates."""

    return F.to_date(
        F.concat(
            F.lit("19"),
            F.col(column_name)
        ),
        "yyyyMMdd"
    )


# Account

cleaned_dfs["account"] = (
    cleaned_dfs["account"]
    .withColumn("account_id", F.col("account_id").cast("long"))
    .withColumn("district_id", F.col("district_id").cast("integer"))
    .withColumn("date", convert_berka_date("date"))
)


# Card

cleaned_dfs["card"] = (
    cleaned_dfs["card"]
    .withColumn("card_id", F.col("card_id").cast("long"))
    .withColumn("disp_id", F.col("disp_id").cast("long"))
    .withColumn(
        "issued",
        F.to_timestamp("issued", "yyMMdd HH:mm:ss")
    )
)


# Client

cleaned_dfs["client"] = (
    cleaned_dfs["client"]
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("district_id", F.col("district_id").cast("integer"))
)


# Disp

cleaned_dfs["disp"] = (
    cleaned_dfs["disp"]
    .withColumn("disp_id", F.col("disp_id").cast("long"))
    .withColumn("client_id", F.col("client_id").cast("long"))
    .withColumn("account_id", F.col("account_id").cast("long"))
)


# District

DISTRICT_INTEGER_COLUMNS = [
    "district_id",
    "population",
    "municipalities_under_500",
    "municipalities_500_to_1999",
    "municipalities_2000_to_9999",
    "municipalities_over_10000",
    "cities"
]

DISTRICT_DECIMAL_COLUMNS = [
    "urban_population_pct",
    "average_salary",
    "unemployment_rate_1995",
    "unemployment_rate_1996",
    "entrepreneurs_per_1000"
]

for column in DISTRICT_INTEGER_COLUMNS:
    cleaned_dfs["district"] = (
        cleaned_dfs["district"]
        .withColumn(column, F.col(column).cast("integer"))
    )

for column in DISTRICT_DECIMAL_COLUMNS:
    cleaned_dfs["district"] = (
        cleaned_dfs["district"]
        .withColumn(column, F.col(column).cast("decimal(12,2)"))
    )


# Loan

cleaned_dfs["loan"] = (
    cleaned_dfs["loan"]
    .withColumn("loan_id", F.col("loan_id").cast("long"))
    .withColumn("account_id", F.col("account_id").cast("long"))
    .withColumn("date", convert_berka_date("date"))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .withColumn("duration", F.col("duration").cast("integer"))
    .withColumn("payments", F.col("payments").cast("decimal(18,2)"))
)


# Orders

cleaned_dfs["orders"] = (
    cleaned_dfs["orders"]
    .withColumn("order_id", F.col("order_id").cast("long"))
    .withColumn("account_id", F.col("account_id").cast("long"))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
)


# Transactions

cleaned_dfs["trans"] = (
    cleaned_dfs["trans"]
    .withColumn("trans_id", F.col("trans_id").cast("long"))
    .withColumn("account_id", F.col("account_id").cast("long"))
    .withColumn("date", convert_berka_date("date"))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .withColumn("balance", F.col("balance").cast("decimal(18,2)"))
)


print("Berka data type conversion completed.")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 9, Finished, Available, Finished, False)

Berka data type conversion completed.


In [8]:
# PaySim data type conversion

cleaned_dfs["paysim"] = (
    cleaned_dfs["paysim"]
    .withColumn("step", F.col("step").cast("integer"))
    .withColumn("amount", F.col("amount").cast("decimal(18,2)"))
    .withColumn(
        "oldbalanceOrg",
        F.col("oldbalanceOrg").cast("decimal(18,2)")
    )
    .withColumn(
        "newbalanceOrig",
        F.col("newbalanceOrig").cast("decimal(18,2)")
    )
    .withColumn(
        "oldbalanceDest",
        F.col("oldbalanceDest").cast("decimal(18,2)")
    )
    .withColumn(
        "newbalanceDest",
        F.col("newbalanceDest").cast("decimal(18,2)")
    )
    .withColumn("isFraud", F.col("isFraud").cast("integer"))
    .withColumn(
        "isFlaggedFraud",
        F.col("isFlaggedFraud").cast("integer")
    )
)


# Verify PaySim schema

cleaned_dfs["paysim"].printSchema()

display(cleaned_dfs["paysim"].limit(5))

print("PaySim data type conversion completed.")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 10, Finished, Available, Finished, False)

root
 |-- step: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- amount: decimal(18,2) (nullable = true)
 |-- nameOrig: string (nullable = true)
 |-- oldbalanceOrg: decimal(18,2) (nullable = true)
 |-- newbalanceOrig: decimal(18,2) (nullable = true)
 |-- nameDest: string (nullable = true)
 |-- oldbalanceDest: decimal(18,2) (nullable = true)
 |-- newbalanceDest: decimal(18,2) (nullable = true)
 |-- isFraud: integer (nullable = true)
 |-- isFlaggedFraud: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, b5337b28-6677-4ce9-900f-1608d17b3579)

PaySim data type conversion completed.


In [9]:
# Document known data quality issues

DATA_QUALITY_NOTES = {
    "district": {
        "column": "unemployment_rate_1995",
        "district_id": 69,
        "decision": "Retain NULL",
        "reason": "Value missing in original source dataset."
    },
    "orders": {
        "column": "k_symbol",
        "decision": "Retain NULL",
        "reason": "Optional transaction classification."
    },
    "trans": {
        "columns": [
            "operation",
            "k_symbol",
            "bank",
            "account"
        ],
        "decision": "Retain NULL",
        "reason": "Missing values form part of the original transaction data."
    }
}



for table_name, issues in DATA_QUALITY_NOTES.items():
    print(f"{table_name.upper()}: {issues}")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 11, Finished, Available, Finished, False)

DISTRICT: {'column': 'unemployment_rate_1995', 'district_id': 69, 'decision': 'Retain NULL', 'reason': 'Value missing in original source dataset.'}
ORDERS: {'column': 'k_symbol', 'decision': 'Retain NULL', 'reason': 'Optional transaction classification.'}
TRANS: {'columns': ['operation', 'k_symbol', 'bank', 'account'], 'decision': 'Retain NULL', 'reason': 'Missing values form part of the original transaction data.'}


### **Define validation rules for the Berka datasets**

In [10]:
# Required fields configuration

REQUIRED_FIELDS = {
    "account": [
        "account_id",
        "district_id",
        "frequency",
        "date"
    ],
    "client": [
        "client_id",
        "birth_number",
        "district_id"
    ],
    "disp": [
        "disp_id",
        "client_id",
        "account_id",
        "type"
    ],
    "trans": [
        "trans_id",
        "account_id",
        "date",
        "type",
        "amount",
        "balance"
    ],
    "paysim": [
        "step",
        "type",
        "amount",
        "nameOrig",
        "nameDest",
        "isFraud",
        "isFlaggedFraud"
    ],
    "card": [
        "card_id",
        "disp_id",
        "type",
        "issued"
    ],
    "district": [
        "district_id",
        "district_name",
        "region",
        "population"
    ],
    "loan": [
        "loan_id",
        "account_id",
        "date",
        "amount",
        "duration",
        "payments",
        "status"
    ],
    "orders": [
        "order_id",
        "account_id",
        "amount"
    ]
}

# Required field validation

def validate_required_fields( df: DataFrame,table_name: str) -> tuple:
    """Separate records with missing required fields."""

    required_columns = REQUIRED_FIELDS[table_name]

    missing_condition = None

    for column in required_columns:

        column_condition = F.col(column).isNull()

        if missing_condition is None:
            missing_condition = column_condition
        else:
            missing_condition = (
                missing_condition | column_condition
            )

    valid_df = df.filter(~missing_condition)

    rejected_df = df.filter(missing_condition)

    print(f"\n{table_name.upper()}")
    print(f"Source rows: {df.count()}")
    print(f"Valid rows: {valid_df.count()}")
    print(f"Rejected rows: {rejected_df.count()}")

    return valid_df, rejected_df


# Validate selected datasets

validation_results = {}

for table_name in REQUIRED_FIELDS:

    valid_df, rejected_df = validate_required_fields(
        cleaned_dfs[table_name],
        table_name
    )

    validation_results[table_name] = {
        "valid": valid_df,
        "rejected": rejected_df
    }

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 12, Finished, Available, Finished, False)


ACCOUNT
Source rows: 4500
Valid rows: 4500
Rejected rows: 0

CLIENT
Source rows: 5369
Valid rows: 5369
Rejected rows: 0

DISP
Source rows: 5369
Valid rows: 5369
Rejected rows: 0

TRANS
Source rows: 1056320
Valid rows: 1056320
Rejected rows: 0

PAYSIM
Source rows: 6362620
Valid rows: 6362620
Rejected rows: 0

CARD
Source rows: 892
Valid rows: 892
Rejected rows: 0

DISTRICT
Source rows: 77
Valid rows: 77
Rejected rows: 0

LOAN
Source rows: 682
Valid rows: 682
Rejected rows: 0

ORDERS
Source rows: 6471
Valid rows: 6471
Rejected rows: 0


### **BUSINESS RULE VALIDATION**

In [11]:
# ==========================================
# PAYSIM RULE VALIDATION
# ==========================================

PAYSIM_VALID_TYPES = [
    "CASH_IN",
    "CASH_OUT",
    "DEBIT",
    "PAYMENT",
    "TRANSFER"
]

paysim_df = cleaned_dfs["paysim"]

paysim_validation = paysim_df.select(
    F.count("*").alias("total_rows"),

    F.sum(
        F.when(F.col("amount") < 0, 1).otherwise(0)
    ).alias("negative_amounts"),

    F.sum(
        F.when(F.col("step") <= 0, 1).otherwise(0)
    ).alias("invalid_steps"),

    F.sum(
        F.when(~F.col("type").isin(PAYSIM_VALID_TYPES), 1).otherwise(0)
    ).alias("invalid_transaction_types"),

    F.sum(
        F.when(~F.col("isFraud").isin(0, 1), 1).otherwise(0)
    ).alias("invalid_fraud_flags"),

    F.sum(
        F.when(~F.col("isFlaggedFraud").isin(0, 1), 1).otherwise(0)
    ).alias("invalid_flagged_fraud_values")
)

display(paysim_validation)

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 13, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 99095ea4-4732-4006-9047-37f680bc5308)

In [12]:
DATA_QUALITY_NOTES["trans_business_rules"] = {
    "non_positive_amounts": {
        "count": 14,
        "decision": "Retain",
        "reason": (
            "Zero-value transactions are associated with "
            "interest and penalty-related transaction symbols."
        )
    },
    "negative_balances": {
        "count": 2999,
        "decision": "Retain",
        "reason": (
            "Negative balances appear in withdrawal and "
            "interest transactions. They may represent "
            "overdraft conditions."
        )
    }
}

for table_name, issues in DATA_QUALITY_NOTES.items():
    print(f"{table_name.upper()}: {issues}")

StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 14, Finished, Available, Finished, False)

DISTRICT: {'column': 'unemployment_rate_1995', 'district_id': 69, 'decision': 'Retain NULL', 'reason': 'Value missing in original source dataset.'}
ORDERS: {'column': 'k_symbol', 'decision': 'Retain NULL', 'reason': 'Optional transaction classification.'}
TRANS: {'columns': ['operation', 'k_symbol', 'bank', 'account'], 'decision': 'Retain NULL', 'reason': 'Missing values form part of the original transaction data.'}
TRANS_BUSINESS_RULES: {'non_positive_amounts': {'count': 14, 'decision': 'Retain', 'reason': 'Zero-value transactions are associated with interest and penalty-related transaction symbols.'}, 'negative_balances': {'count': 2999, 'decision': 'Retain', 'reason': 'Negative balances appear in withdrawal and interest transactions. They may represent overdraft conditions.'}}


In [13]:

# ==========================================
# CUSTOMER DATA PROTECTION
# ==========================================

# Create a separate collection for Silver transformations
silver_dfs = cleaned_dfs.copy()

# Remove the encoded personal identifier
silver_dfs["client"] = (
    silver_dfs["client"]
    .drop("birth_number")
)

# Verify the updated Client dataset
print("Silver Client schema:")
silver_dfs["client"].printSchema()

print(
    f"Client records retained: "
    f"{silver_dfs['client'].count()}"
)

print(
    "birth_number present:",
    "birth_number" in silver_dfs["client"].columns
)


StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 15, Finished, Available, Finished, False)

Silver Client schema:
root
 |-- client_id: long (nullable = true)
 |-- district_id: integer (nullable = true)

Client records retained: 5369
birth_number present: False


In [14]:

# ==========================================
# CLEAN DISTRICT CRIME COLUMNS
# ==========================================

CRIME_COLUMNS = [
    "crimes_1995",
    "crimes_1996"
]

for column in CRIME_COLUMNS:

    silver_dfs["district"] = (
        silver_dfs["district"]
        .withColumn(
            column,
            F.when(
                F.trim(F.col(column)) == "?",
                F.lit(None).cast("string")
            )
            .otherwise(F.trim(F.col(column)))
            .cast("integer")
        )
    )

# Verify the updated schema
silver_dfs["district"].printSchema()

# Count missing values in crime columns
crime_quality = silver_dfs["district"].select(
    F.count("*").alias("total_districts"),
    F.sum(
        F.when(F.col("crimes_1995").isNull(), 1).otherwise(0)
    ).alias("missing_crimes_1995"),
    F.sum(
        F.when(F.col("crimes_1996").isNull(), 1).otherwise(0)
    ).alias("missing_crimes_1996")
)

display(crime_quality)


StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 16, Finished, Available, Finished, False)

root
 |-- district_id: integer (nullable = true)
 |-- district_name: string (nullable = true)
 |-- region: string (nullable = true)
 |-- population: integer (nullable = true)
 |-- municipalities_under_500: integer (nullable = true)
 |-- municipalities_500_to_1999: integer (nullable = true)
 |-- municipalities_2000_to_9999: integer (nullable = true)
 |-- municipalities_over_10000: integer (nullable = true)
 |-- cities: integer (nullable = true)
 |-- urban_population_pct: decimal(12,2) (nullable = true)
 |-- average_salary: decimal(12,2) (nullable = true)
 |-- unemployment_rate_1995: decimal(12,2) (nullable = true)
 |-- unemployment_rate_1996: decimal(12,2) (nullable = true)
 |-- entrepreneurs_per_1000: decimal(12,2) (nullable = true)
 |-- crimes_1995: integer (nullable = true)
 |-- crimes_1996: integer (nullable = true)



SynapseWidget(Synapse.DataFrame, b58b683f-ea85-4977-93ed-cf85d6a70e56)

### **DATASET VALIDATION**

In [15]:

# ==========================================
# SILVER DATASET VALIDATION
# ==========================================

silver_validation_results = []

for table_name, df in silver_dfs.items():

    row_count = df.count()
    column_count = len(df.columns)

    silver_validation_results.append((
        table_name,
        row_count,
        column_count
    ))

    print(
        f"{table_name.upper()}: "
        f"{row_count:,} rows, "
        f"{column_count} columns"
    )


# Create validation summary
silver_summary = spark.createDataFrame(
    silver_validation_results,
    [
        "table_name",
        "row_count",
        "column_count"
    ]
)

display(silver_summary)


StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 17, Finished, Available, Finished, False)

ACCOUNT: 4,500 rows, 4 columns
CARD: 892 rows, 4 columns
CLIENT: 5,369 rows, 2 columns
DISP: 5,369 rows, 4 columns
DISTRICT: 77 rows, 16 columns
LOAN: 682 rows, 7 columns
ORDERS: 6,471 rows, 6 columns
TRANS: 1,056,320 rows, 10 columns
PAYSIM: 6,362,620 rows, 11 columns


SynapseWidget(Synapse.DataFrame, 654d68a9-b531-4252-8b5d-74b6d0a84d4a)

### **SAVE SILVER DELTA TABLES**

In [16]:

# ==========================================
# SAVE SILVER DELTA TABLES
# ==========================================

SILVER_TABLES = {
    "account": "silver_account",
    "card": "silver_card",
    "client": "silver_client",
    "disp": "silver_disp",
    "district": "silver_district",
    "loan": "silver_loan",
    "orders": "silver_orders",
    "trans": "silver_trans",
    "paysim": "silver_paysim"
}


def save_silver_table(
    df: DataFrame,
    table_name: str
) -> None:
    """Save a transformed DataFrame as a Delta table."""

    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )

    print(f"Saved Delta table: {table_name}")


# Save all nine Silver datasets
for dataset_name, table_name in SILVER_TABLES.items():

    save_silver_table(
        silver_dfs[dataset_name],
        table_name
    )

print("Silver layer storage completed.")


StatementMeta(, d4101760-a9cf-47a0-a41f-506f1a12944e, 18, Finished, Available, Finished, False)

Saved Delta table: silver_account
Saved Delta table: silver_card
Saved Delta table: silver_client
Saved Delta table: silver_disp
Saved Delta table: silver_district
Saved Delta table: silver_loan
Saved Delta table: silver_orders
Saved Delta table: silver_trans
Saved Delta table: silver_paysim
Silver layer storage completed.
